In [0]:
WITH demanda AS (

    SELECT
        segmento,
        item_principal_cadeia,
        uf_cliente,
        centro_distribuicao_original AS centro_original,

        CASE
            WHEN uf_cliente = 'TO' AND centro_distribuicao_original = '0203' THEN '0209'
            WHEN uf_cliente = 'TO' AND centro_distribuicao_original = '0503' THEN '0505'
            WHEN uf_cliente = 'AP' AND centro_distribuicao_original = '0209' THEN '0232'
            ELSE centro_distribuicao_original
        END AS centro_ajustado,

        date_format(data_ordem, 'yyyy/MM') AS ano_mes,
        quantidade

    FROM `parts_hdbk_sandbox`.`pr_demand`.`demand_analytical_base`

    WHERE organizacao_vendas = '0200'
      AND data_ordem >= '2026-04-01'
),

demanda_pivot AS (

    SELECT *

    FROM demanda

    PIVOT (
        SUM(quantidade)
        FOR ano_mes IN (
            '2026/04' AS `2026/04`,
            '2026/05' AS `2026/05`,
            '2026/06' AS `2026/06`,
            '2026/07' AS `2026/07`,
            '2026/08' AS `2026/08`,
            '2026/09' AS `2026/09`
        )
    )
)

SELECT
    segmento,
    item_principal_cadeia,
    uf_cliente,
    centro_original,
    centro_ajustado,
    
    COALESCE(`2026/04`, 0) AS `2026/04`,
    COALESCE(`2026/05`, 0) AS `2026/05`,
    COALESCE(`2026/06`, 0) AS `2026/06`,
    COALESCE(`2026/07`, 0) AS `2026/07`,
    COALESCE(`2026/08`, 0) AS `2026/08`,
    COALESCE(`2026/09`, 0) AS `2026/09`

FROM demanda_pivot

ORDER BY item_principal_cadeia;